In [32]:
import pandas as pd


# Q1 - Build Knowledge Base

roll_number = "1024160030"
fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]


In [33]:
categories = ["billing", "account", "general"]
last_two_digits = roll_number[-2:]
personalized_entries = []

for digit in last_two_digits:
    d = int(digit)
    category = categories[d % 3]

    if d == 3:
        entry = {
            "question": "how can i get a fee receipt",
            "answer": "You can download your fee receipt after successful payment.",
            "keywords": "fee receipt payment",
            "category": category
        }
    else:
        entry = {
            "question": "can i pay my fee online",
            "answer": "Yes, you can pay your fee online using UPI or net banking.",
            "keywords": "fee online payment",
            "category": category
        }

    personalized_entries.append(entry)

all_entries = fixed_entries + personalized_entries

df = pd.DataFrame(all_entries)

print("Q1 - Final 6-row DataFrame:")
df


Q1 - Final 6-row DataFrame:


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how can i get a fee receipt,You can download your fee receipt after succes...,fee receipt payment,billing
5,can i pay my fee online,"Yes, you can pay your fee online using UPI or ...",fee online payment,billing


In [34]:
# Q2 - Generate and Score

def score_query(query, df):
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        keywords = set(row["keywords"].lower().split())
        question_words = set(row["question"].lower().split())

        keyword_matches = len(query_words.intersection(keywords))
        question_matches = len(query_words.intersection(question_words))

        score = keyword_matches + question_matches

        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })

    results.sort(key=lambda x: x["score"], reverse=True)

    return results


query = input("\nEnter a query: ")

results = score_query(query, df)

print("\nQ2 - Matching entries ranked by confidence:")

if len(results) == 0:
    print("No matching entries found.")
else:
    for result in results:
        print("Question:", result["question"])
        print("Answer:", result["answer"])
        print("Category:", result["category"])
        print("Confidence Score:", result["score"])
        print()



Enter a query: Login

Q2 - Matching entries ranked by confidence:
Question: how to reset password
Answer: Go to Settings > Reset Password.
Category: account
Confidence Score: 1



In [35]:
# Q3 - Same Category

def same_category(category_name, df):
    return df[df["category"] == category_name][["question", "answer"]]

personalized_category = personalized_entries[0]["category"]

print("Q3 - Questions in category:", personalized_category)
print(same_category(personalized_category, df))


Q3 - Questions in category: billing
                      question  \
0       what is the annual fee   
3        how can i pay the fee   
4  how can i get a fee receipt   
5      can i pay my fee online   

                                              answer  
0                          The annual fee is Rs 500.  
3         You can pay via UPI, card, or net banking.  
4  You can download your fee receipt after succes...  
5  Yes, you can pay your fee online using UPI or ...  


In [36]:
# Q4 - Add New Keyword

print("\nQ4 - Add a new keyword")

new_keyword = input("Enter a new keyword: ")

df.loc[0, "keywords"] = df.loc[0, "keywords"] + " " + new_keyword

file_name = roll_number + "_faq_data.csv"

df.to_csv(file_name, index=False)

print("Updated DataFrame:")
print(df)

print("\nData saved to:", file_name)


Q4 - Add a new keyword
Enter a new keyword: Purpose
Updated DataFrame:
                      question  \
0       what is the annual fee   
1        how to reset password   
2  what are your working hours   
3        how can i pay the fee   
4  how can i get a fee receipt   
5      can i pay my fee online   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  You can download your fee receipt after succes...   
5  Yes, you can pay your fee online using UPI or ...   

                        keywords category  
0  fee cost price charge Purpose  billing  
1           password reset login  account  
2         hours timing open time  general  
3            pay payment upi fee  billing  
4            fee receipt payment  billing  
5             fee online payment

In [37]:
# Q5 - Count Entries

print("\nQ5 - Number of FAQ entries per category:")
category_counts = df.groupby("category").size()
print(category_counts)



Q5 - Number of FAQ entries per category:
category
account    1
billing    4
general    1
dtype: int64


In [38]:

# Q6 - Tie Handling

def score_query_with_ties(query, df):
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        keywords = set(row["keywords"].lower().split())
        question_words = set(row["question"].lower().split())

        keyword_matches = len(query_words.intersection(keywords))
        question_matches = len(query_words.intersection(question_words))

        score = keyword_matches + question_matches

        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })

    if len(results) == 0:
        print("No matching entries found.")
        return

    results.sort(key=lambda x: x["score"], reverse=True)

    highest_score = results[0]["score"]

    top_matches = []

    for result in results:
        if result["score"] == highest_score:
            top_matches.append(result)

    print("\nHighest confidence score:", highest_score)

    if len(top_matches) > 1:
        print("Tie found! All equally matching entries are:")

        for result in top_matches:
            print("\nQuestion:", result["question"])
            print("Answer:", result["answer"])
            print("Category:", result["category"])
            print("Score:", result["score"])

    else:
        print("No tie found. Best matching entry is:")

        result = top_matches[0]

        print("Question:", result["question"])
        print("Answer:", result["answer"])
        print("Category:", result["category"])
        print("Score:", result["score"])


# Demonstration with a query that produces a tie
print("\nQ6 - Tie Demonstration")

print("\nTie Query:")
score_query_with_ties("fee", df)

print("\nNon-Tie Query:")
score_query_with_ties("password", df)


Q6 - Tie Demonstration

Tie Query:

Highest confidence score: 2
Tie found! All equally matching entries are:

Question: what is the annual fee
Answer: The annual fee is Rs 500.
Category: billing
Score: 2

Question: how can i pay the fee
Answer: You can pay via UPI, card, or net banking.
Category: billing
Score: 2

Question: how can i get a fee receipt
Answer: You can download your fee receipt after successful payment.
Category: billing
Score: 2

Question: can i pay my fee online
Answer: Yes, you can pay your fee online using UPI or net banking.
Category: billing
Score: 2

Non-Tie Query:

Highest confidence score: 2
No tie found. Best matching entry is:
Question: how to reset password
Answer: Go to Settings > Reset Password.
Category: account
Score: 2
